# Multi-Agent Research Assistant — Colab / Binder notebook

A zero-install, runnable version of the course's [Multi-Agent Research Assistant project](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/multi-agent-research) and its companion script, [`examples/multi-agent-research/agent.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/add-multi-agent-research-project/examples/multi-agent-research/agent.py).

Three small, narrowly-instructed sub-agents — built with LangChain's [`deepagents`](https://github.com/langchain-ai/deepagents) — cooperate on one research question:

- **`planner`** breaks the question into 3-5 focused sub-questions.
- **`researcher`** answers each sub-question on its own, one at a time.
- **`writer`** synthesizes every sub-question/answer pair into one coherent final report.

**Honesty note:** the `researcher` sub-agent answers from the model's own training knowledge — there is no real web search tool wired in here. That keeps this example small and free-tier friendly, but it means answers can be stale or wrong on anything the model wasn't trained on well.

This notebook runs entirely on a free-tier LLM API — no GPU needed, no local setup. You just need a free API key from one of six supported providers (picked below).

## 1. Install dependencies

This installs `deepagents` plus the LangChain client packages for all six supported providers, so you can pick any of them below without re-running this cell.

In [ ]:
!pip install -q "deepagents>=0.6.12" "langchain-openai>=1.3.5" "langchain-google-genai>=4.2.7" \

    "langchain-groq>=1.1.3" "langchain-mistralai>=1.1.6" "python-dotenv>=1.2.2"

## 2. Pick a provider and enter your API key

Pick any of the six free-tier providers this project supports. Since Colab/Binder sessions are ephemeral (no local `.env` file persists), we collect the key interactively with `getpass` instead — it's never displayed or saved to disk, and only lives in this session's memory.

| `PROVIDER` value | Where to get a free key |
|---|---|
| `github` *(default)* | https://github.com/settings/tokens (personal access token, `models: read` scope) |
| `gemini` | https://aistudio.google.com/ |
| `groq` | https://console.groq.com/keys |
| `mistral` | https://console.mistral.ai/api-keys |
| `cerebras` | https://cloud.cerebras.ai/ |
| `openrouter` | https://openrouter.ai/keys |

In [ ]:
import os

from getpass import getpass



# Change this to whichever provider you have a free-tier key for.

PROVIDER = "github"  # one of: github, gemini, groq, mistral, cerebras, openrouter



_ENV_VAR = {

    "github": "GITHUB_TOKEN",

    "gemini": "GOOGLE_API_KEY",

    "groq": "GROQ_API_KEY",

    "mistral": "MISTRAL_API_KEY",

    "cerebras": "CEREBRAS_API_KEY",

    "openrouter": "OPENROUTER_API_KEY",

}[PROVIDER]



if not os.environ.get(_ENV_VAR):

    os.environ[_ENV_VAR] = getpass(f"Enter your {PROVIDER} API key ({_ENV_VAR}): ")



print(f"{_ENV_VAR} is set for this session.")

## 3. Build the LLM for the chosen provider

Same six provider builders as `agent.py` in the course repo — each one returns a LangChain chat model instance reading its key from the environment variable set above.

In [ ]:
def _build_github_model():

    from langchain_openai import ChatOpenAI

    return ChatOpenAI(

        model="gpt-4o-mini",

        api_key=os.environ["GITHUB_TOKEN"],

        base_url="https://models.github.ai/inference",

    )





def _build_gemini_model():

    from langchain_google_genai import ChatGoogleGenerativeAI

    return ChatGoogleGenerativeAI(

        model="gemini-3.5-flash",

        google_api_key=os.environ["GOOGLE_API_KEY"],

    )





def _build_groq_model():

    from langchain_groq import ChatGroq

    return ChatGroq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])





def _build_mistral_model():

    from langchain_mistralai import ChatMistralAI

    return ChatMistralAI(model="mistral-small-latest", api_key=os.environ["MISTRAL_API_KEY"])





def _build_cerebras_model():

    from langchain_openai import ChatOpenAI

    return ChatOpenAI(

        model="llama-3.3-70b",

        api_key=os.environ["CEREBRAS_API_KEY"],

        base_url="https://api.cerebras.ai/v1",

    )





def _build_openrouter_model():

    from langchain_openai import ChatOpenAI

    return ChatOpenAI(

        model="meta-llama/llama-3.3-70b-instruct:free",

        api_key=os.environ["OPENROUTER_API_KEY"],

        base_url="https://openrouter.ai/api/v1",

    )





PROVIDERS = {

    "github": _build_github_model,

    "gemini": _build_gemini_model,

    "groq": _build_groq_model,

    "mistral": _build_mistral_model,

    "cerebras": _build_cerebras_model,

    "openrouter": _build_openrouter_model,

}



model = PROVIDERS[PROVIDER]()

print(f"Model built for provider: {PROVIDER}")

## 4. Define the planner, researcher, and writer sub-agents

Each sub-agent in `deepagents` is just a plain dict: a `name`, a `description` (used by the top-level agent to decide when to delegate to it), and its own narrow `system_prompt`.

In [ ]:
planner_subagent = {

    "name": "planner",

    "description": "Breaks a research question down into 3-5 focused, independently-answerable sub-questions.",

    "system_prompt": (

        "You are a research planner. Given a broad research question, break it "

        "into 3 to 5 specific, independently-answerable sub-questions that together "

        "cover the topic well. Output ONLY a numbered list of sub-questions -- no "

        "preamble, no answers, just the questions themselves."

    ),

}



researcher_subagent = {

    "name": "researcher",

    "description": "Answers one specific sub-question at a time, concisely and factually.",

    "system_prompt": (

        "You are a researcher. Answer the single sub-question you are given as "

        "accurately and concisely as you can, using your own knowledge. You have "

        "no web search tool in this version -- if you are not confident about a "

        "fact, say so explicitly rather than guessing. Answer in 2-4 sentences."

    ),

}



writer_subagent = {

    "name": "writer",

    "description": "Synthesizes a set of sub-question answers into one coherent final report.",

    "system_prompt": (

        "You are a writer. Given a research question and a set of sub-question/answer "

        "pairs, synthesize them into one coherent, well-organized report of a few "

        "paragraphs. Do not just concatenate the answers -- connect them into prose "

        "that reads as a single piece of writing, and note plainly if the underlying "

        "research flagged low confidence anywhere."

    ),

}

## 5. Wire the sub-agents together

The top-level agent doesn't do any research itself — its whole job is delegation, strictly in order: plan, then research each sub-question, then write. `subagents=[...]` is the whole mechanism: the top-level agent sees each sub-agent's `name` and `description` the same way it would see a tool's name and docstring, and decides when to hand off to which one.

In [ ]:
from deepagents import create_deep_agent



TOP_LEVEL_SYSTEM_PROMPT = (

    "You coordinate a research task using your sub-agents, strictly in this order: "

    "1) delegate to the 'planner' sub-agent to get a numbered list of sub-questions. "

    "2) delegate each sub-question, one at a time, to the 'researcher' sub-agent. "

    "3) delegate to the 'writer' sub-agent, giving it the original question plus every "

    "sub-question/answer pair, and have it produce the final report. "

    "Return ONLY the writer's final report as your answer -- no intermediate steps."

)



agent = create_deep_agent(

    model=model,

    subagents=[planner_subagent, researcher_subagent, writer_subagent],

    system_prompt=TOP_LEVEL_SYSTEM_PROMPT,

)

print("Agent built.")

## 6. Helper functions: rate-limit retry and readable output

One research question here costs at least one planner call, one researcher call per sub-question (typically 3-5), and one writer call — six to eight round trips minimum, so hitting a free-tier rate limit is more likely here than in a single-agent example. `ask()` catches a 429/rate-limit error, waits, and retries once before giving up on the question.

In [ ]:
import re

import time



from langchain_core.messages import AIMessage, HumanMessage, ToolMessage



_RATE_LIMIT_SIGNALS = ("429", "RESOURCE_EXHAUSTED", "rate_limit", "Too Many Requests", "rate limit")





def _retry_delay_seconds(error_message: str):

    """Parse "Please retry in 41.7s" (or similar) out of a provider's error message, if present."""

    match = re.search(r"retry in ([\d.]+)s", error_message)

    return float(match.group(1)) if match else None





def ask(agent, question: str, max_retries: int = 1):

    """Run one research question through the pipeline and return the raw LangGraph result."""

    try:

        return agent.invoke({"messages": [HumanMessage(content=question)]})

    except Exception as error:

        message = str(error)

        if not any(signal in message for signal in _RATE_LIMIT_SIGNALS):

            raise  # a real bug, not a rate limit -- don't hide it

        delay = _retry_delay_seconds(message) or 30.0

        if max_retries <= 0:

            print(f"\u26a0\ufe0f  Rate limited and out of retries -- giving up on this question. ({message[:200]})")

            return None

        print(f"\u26a0\ufe0f  Rate limited by the free tier. Waiting {delay:.0f}s before retrying...")

        time.sleep(delay)

        return ask(agent, question, max_retries=max_retries - 1)





def print_conversation(result: dict) -> None:

    """Pretty-print an agent result as a readable step-by-step trace."""

    for message in result["messages"]:

        if isinstance(message, HumanMessage):

            print(f"\U0001f9d1 You: {message.content}")

        elif isinstance(message, ToolMessage):

            content = str(message.content)

            if len(content) > 300:

                content = content[:300] + "\u2026"

            print(f"\U0001f527 Sub-agent result ({message.name}): {content}")

        elif isinstance(message, AIMessage):

            for call in message.tool_calls:

                print(f"\U0001f916 Agent \u2192 delegating to {call['name']}({call['args']})")

            text = message.content

            if isinstance(text, list):  # Gemini sometimes returns structured content blocks

                text = "".join(block.get("text", "") for block in text if isinstance(block, dict))

            if text:

                print(f"\U0001f916 Agent: {text}")





def final_answer(result: dict) -> str:

    """Just the writer's final report text -- useful when you don't need the full trace."""

    last = result["messages"][-1]

    content = last.content

    if isinstance(content, list):

        return "".join(block.get("text", "") for block in content if isinstance(block, dict))

    return str(content)

## 7. Run the pipeline on a sample research question

This asks the pipeline one example research question, prints a readable step-by-step trace of every delegation, then prints the writer's final synthesized report. Expect this to take a minute or two — it's several round trips through a free-tier model. Feel free to change `question` to whatever you're curious about.

In [ ]:
question = "What makes a programming language good for beginners to learn first?"



print("=" * 70)

result = ask(agent, question)

if result is not None:

    print_conversation(result)

    print()

    print("Final report:")

    print(final_answer(result))